# Lab 5 — Handwritten Digit Recognition with an RNN (LSTM)
### Deep Learning Laboratory (23DS5PCDLG) · Student Fill-in Notebook

**Big idea.** We read each `28 x 28` digit image as a **sequence of 28 rows** (28 time steps),
where each row is a vector of 28 pixel values (28 features). An **LSTM** reads the rows one by
one, keeps a memory, and after the last row predicts the digit (0–9).

---
### How to use this notebook
Wherever you see a blank marked `None  # TODO (Qn)` or `# TODO`, **replace it** with the correct
value or line of code. Each task has a hint. Run the cells top to bottom. When everything is
filled correctly, the model should reach **~98% test accuracy** in 10 epochs.

> Do **not** change cells marked *(given — do not edit)*. Fill only the `TODO` blanks.


## Step 0 — Imports *(given — do not edit)*

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## Task 1 — Load and preprocess the data

MNIST gives you `x_train` of shape `(60000, 28, 28)` with pixel values `0–255`, and integer
labels `y_train` in `0–9`. You must:

1. **Normalise** the pixels to the range `[0, 1]`.  *Hint: divide by the maximum pixel value.*
2. **Keep the sequence shape** `(samples, time_steps, features) = (n, 28, 28)`.
   *Hint: it is already `(n, 28, 28)` — state the reshape explicitly so the intent is clear.*
3. **One-hot encode** the labels into 10-dimensional vectors.
   *Hint: how many digit classes are there?*


In [ ]:
# --- load (given) ---
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print("raw shapes:", x_train.shape, y_train.shape)

# Q1a) Normalise pixels to [0, 1]
PIXEL_MAX = 255            # largest pixel value
x_train = x_train / PIXEL_MAX
x_test  = x_test  / PIXEL_MAX

# Q1b) Reshape to (samples, time_steps, features)
TIME_STEPS = 28
FEATURES   = 28
x_train = x_train.reshape((x_train.shape[0], TIME_STEPS, FEATURES))
x_test  = x_test.reshape((x_test.shape[0],  TIME_STEPS, FEATURES))

# Q1c) One-hot encode the labels
NUM_CLASSES = 10
y_train = to_categorical(y_train, NUM_CLASSES)
y_test  = to_categorical(y_test,  NUM_CLASSES)

print("prepared:", x_train.shape, y_train.shape)

#### Quick check *(given — run to verify Task 1)*

In [ ]:
assert x_train.max() <= 1.0 and x_train.min() >= 0.0, "pixels not in [0,1]"
assert x_train.shape[1:] == (28, 28), "sequence shape should be (28, 28)"
assert y_train.shape[1] == 10, "labels should be one-hot with 10 columns"
print("Task 1 looks correct ✔")

## Task 2 — Build the RNN (LSTM) model

Build a `Sequential` model with:

1. An **LSTM layer** with **128 memory units** that reads the 28-step sequence.
   *Hint: `input_shape` is `(time_steps, features)` — NOT the batch size.*
2. A **Dense output layer** with one neuron per class and a **softmax** activation
   (so the outputs are class probabilities that sum to 1).

> Note on activation: the lab uses `activation='relu'` inside the LSTM. The **standard/
> recommended** choice is the default `'tanh'` (it also enables the fast cuDNN GPU kernel).
> You may try either and compare.


In [ ]:
model = models.Sequential()

# Q2a) LSTM layer
LSTM_UNITS = 128
model.add(layers.LSTM(LSTM_UNITS,
                      activation='relu',
                      input_shape=(28, 28)))

# Q2b) Output layer: one neuron per class, softmax activation
OUT_UNITS = 10
model.add(layers.Dense(OUT_UNITS, activation='softmax'))

model.summary()

**Reflection Q2c.** Read the LSTM layer's parameter count from `model.summary()`, then work it
out by hand and check the two match.

For an LSTM, `params = 4 × [ (features + units) × units + units ]`.

> **Your answer (Q2c):** params = 4 × [ (28 + 128) × 128 + 128 ] = **80,384**

## Task 3 — Compile, train, evaluate

1. **Compile** with the `adam` optimiser, `categorical_crossentropy` loss (because labels are
   one-hot), and track `accuracy`.
2. **Train** for 10 epochs with a batch size of 64, using the test set for validation.
3. **Evaluate** on the test set and print the accuracy.


In [ ]:
# Q3a) Compile
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Q3b) Train
history = model.fit(x_train, y_train,
                    epochs=10,
                    batch_size=64,
                    validation_data=(x_test, y_test))

# Q3c) Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc * 100:.2f}%")

## Task 4 — Visualise the training *(mostly given — fill one blank)*

Plot the training and validation accuracy curves to see learning progress.


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.legend(); plt.title('LSTM on MNIST')
plt.tight_layout(); plt.show()

## Task 5 — Short-answer questions

1. **Why do we feed the image as `(28, 28)` instead of flattening it to a 784-vector as in an MLP?**

   The LSTM takes the image one row at a time. So, 28 rows become 28 time steps, and each row has 28 pixel values. Keeping this shape lets the model use the order of the rows. If we flatten the image, it is no longer given as a sequence.

2. **What is the role of the hidden state `h_t` versus the cell state `C_t` in the LSTM?**

   The hidden state `h_t` is the output at the current time step and is passed forward. The cell state `C_t` carries information through the sequence, helping the LSTM remember things for longer. The gates decide what information should be kept or changed.

3. **The final `Dense` layer uses softmax. Why softmax and not sigmoid here?**

   There are 10 possible answers, but each image is only one digit. Softmax gives a probability for each digit, and all the probabilities add up to 1. Sigmoid gives separate probabilities, so it is more useful when multiple labels can be correct at the same time.

4. **If you changed `LSTM(128)` to `LSTM(32)`, what would you expect to happen to accuracy and to the parameter count? Why?**

   The number of parameters would go down from 80,384 to 7,808 because there are fewer LSTM units. Accuracy might also go down because the model has less capacity to learn the patterns. It could still work well, though, so the actual accuracy would have to be checked by training it.

5. **Replace `activation='relu'` with the default `'tanh'`. Does the accuracy change? Which is recommended, and why?**

   The accuracy could change a little when using `tanh` instead of `relu`. We would need to train both versions to know the difference. `tanh` is the usual LSTM activation and is generally recommended. It can also let TensorFlow use its faster cuDNN implementation when the other requirements are met.
